# 15 Normal Failure Diagnosis and Candidate Policy v2

Diagnostic-only step for the failed Hijack- and Leak-associated Normal controls. This notebook does not relabel failed controls, download replacement UPDATE files, access Validation/Holdout, train models, add CAIDA/RPKI, or modify frozen roster/partition files.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

diagnosis = pd.read_csv(PILOT / 'normal_failure_diagnosis_v1.csv')
collector_visibility = pd.read_csv(PILOT / 'normal_collector_visibility_v1.csv')
replacements = pd.read_csv(PILOT / 'normal_replacement_candidates_v1.csv')
rib_qualification = pd.read_csv(PILOT / 'normal_rib_update_qualification_results_v1.csv')
method_comparison = pd.read_csv(PILOT / 'normal_method_comparison_v1.csv')

## Failure Diagnosis

In [ ]:
assert len(diagnosis) == 4
diagnosis[['candidate_period_id', 'category', 'rib_before_visible', 'rib_after_visible', 'relevant_update_count', 'suspicious_update_count', 'stable_window_count', 'failure_type', 'failure_class', 'recommended_action']]

In [ ]:
diagnosis['failure_class'].value_counts()

## Observability vs Non-Normal Behavior

Insufficient observability is not evidence of abnormal routing. It means the candidate lacks positive RouteViews evidence and must remain `UNKNOWN` unless another historically available collector confirms the route state.

In [ ]:
diagnosis.groupby(['failure_class', 'failure_type']).size()

## Collector Visibility Checks

Alternate collector rows use lightweight historical RIB availability checks only. They are not treated as ASN/prefix-visible until a RIB visibility parse confirms observability.

In [ ]:
collector_visibility.groupby(['collector', 'historical_availability']).size().unstack(fill_value=0)

In [ ]:
collector_visibility[['candidate_period_id', 'collector', 'historical_availability', 'asn_visible', 'prefix_visible', 'estimated_rib_bytes']].head(20)

## Replacement Candidate Pre-Screen

In [ ]:
replacements['pre_screen_status'].value_counts()

In [ ]:
replacements[['associated_incident_uid', 'category', 'candidate_start', 'candidate_end', 'days_from_incident', 'collector', 'known_incident_overlap', 'pre_screen_status', 'estimated_update_file_count']]

## Methodological Implication

In [ ]:
method_comparison

## Policy v2 Verdict

The Normal candidate policy should add RIB visibility as a mandatory pre-screen before UPDATE acquisition. Calendar offset alone is not sufficient, and short incident-derived candidate periods cannot satisfy a 30-minute sustained-stability requirement. `route-views2` should remain the preferred first collector, but it is not sufficient as the only collector across diverse networks; alternate collectors should be selected deterministically by historical availability and ASN/prefix visibility only.